In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_path = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages from PDF")
print("\n- First Page preview (first 500 char) -")
print(pages[6].page_content[:500])

Loaded 9 pages from PDF

- First Page preview (first 500 char) -
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP
Multimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of
legacy calls), fast


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100,
    separators = ["\n\n", "\n", ".", " "], 
)

chunks = splitter.split_documents(pages)

len(chunks)

33

In [11]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [12]:
chunks[1].page_content

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app\ndownloads at speeds of 1-14 Mbps. This generation established the foundation for smartphone adoption\nworldwide.'

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored")

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 9317.06it/s]


Vector store ready. 33 vectors stored


In [20]:
retrivers = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE? How can I imporve the call quality?"
retrivered = retrivers.invoke(test_query)

for i, doc in enumerate (retrivered, 1):
    print(f" - chunk {i} - ")
    print(doc.page_content[:300])
    print ()

 - chunk 1 - 
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

 - chunk 2 - 
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

 - chunk 3 - 
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for example, because the called party is on a network
that does not support VoLTE interconnect  - the network automatically falls back to a 3G circuit-switched call. This
fallback is transparent to the customer but results in lower a



In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_groq import ChatGroq

#next line is where two chunks join
def format_doc(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

System_prompt = """\
    Act as an helpful telecom assistance.
    Answer the question from only the context provided below. If it's not in context communicate to the user clearly.

    Context:
    {context}

"""

prompt = ChatPromptTemplate.from_messages([
        ("system", System_prompt),
        ("human", "{question}"),
])

llm = ChatGroq (
    model = "openai/gpt-oss-120b",
    temperature = 0,
    reasoning_format = "parsed",
)

chain = (
    {"context": retrivers | format_doc, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("rag cahin assembled")

rag cahin assembled


In [46]:
question = "How does internation work"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does internation work

A: International roaming works by connecting a subscriber’s device to a partner (visited) network when the user is outside the coverage area of their home network.  

1. **Network connection** – The device automatically attaches to the visited‑country operator’s network.  

2. **Authentication** – The visited network authenticates the subscriber using an inter‑operator signalling protocol such as **SS7** or **Diameter**.  

3. **Authorization** – The home network receives the authentication request, validates the subscriber’s subscription, and authorises the requested services (voice, SMS, data).  

4. **Traffic handling** – All voice, SMS, and data traffic is **tunnelled back to the home network** so that the home operator can apply its own rate plan and generate billing records. This tunnelling introduces a bit more latency compared with using a purely local network.  

In short, roaming relies on a handshake between the visited and home operators (via S

In [47]:
question = "How does internation work? Can you tell me how amount will I be charged per minute?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does internation work? Can you tell me how amount will I be charged per minute?

A: I’m sorry—I don’t have the specific per‑minute rate for international calls in the information you provided.  

What I can tell you from the context you shared is that the charge you see for any call (including international) depends on a few factors:

| Factor | How it affects the charge |
|--------|---------------------------|
| **Rate plan** | Your plan (prepaid, post‑paid, hybrid, or a bundle) defines the base price for voice minutes, including any international allowances. |
| **Bundle activation** | If you have an international‑calling bundle, the system records the exact timestamp when the bundle becomes active. Charges are applied only after that time. |
| **Roaming** | If you’re making the call while roaming, the system checks whether a roaming bundle is active. If not, you may see an over‑charge, and the itemised bill will show the timestamp of the first roaming event versus the bundle 